In [2]:
import pandas as pd
import os
import ast

os.makedirs('clean_weather_csv', exist_ok=True)

weather_dir = 'weather_csv'
for filename in os.listdir(weather_dir):
    if filename.endswith('.csv'):
        input_path = os.path.join(weather_dir, filename)
        output_filename = filename.replace('.csv', 'clean.csv')
        output_path = os.path.join('clean_weather_csv', output_filename)
        
        df = pd.read_csv(input_path)
        if 'dts' in df.columns:
            records = []
            for _, row in df.iterrows():
                station_id = row['StationID']
                try:
                    data_list = ast.literal_eval(row['dts'])
                    
                    for item in data_list:
                        record = {
                            'StationID': station_id,
                            'DataTime': item.get('DataTime'),
                            'AirTemperature': item.get('AirTemperature', {}).get('Instantaneous'),
                            'RelativeHumidity': item.get('RelativeHumidity', {}).get('Instantaneous'),
                            'Precipitation': item.get('Precipitation', {}).get('Accumulation')
                        }
                        records.append(record)
                except:
                    pass
            
            if records:
                result_df = pd.DataFrame(records)
                result_df.to_csv(output_path, index=False)


In [3]:
import pandas as pd
import os
import numpy as np

clean_dir = 'clean_weather_csv'
for filename in os.listdir(clean_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(clean_dir, filename)
        
        df = pd.read_csv(file_path)
        
        numeric_columns = ['AirTemperature', 'RelativeHumidity', 'Precipitation']
        for col in numeric_columns:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                df.loc[df[col] < -50, col] = np.nan
        
        df.to_csv(file_path, index=False)
